## Concept 9 — CRAG (Corrective RAG)

### What is it?
All previous concepts assume your knowledge base has the answer.
CRAG adds a **quality gate** — it evaluates whether retrieved docs are actually relevant.
If docs are poor quality, it falls back to a web search.

### Pipeline
```
Query
  → Retrieve from knowledge base
  → Evaluator: score each doc (RELEVANT / IRRELEVANT / AMBIGUOUS)
  ↓
  RELEVANT           IRRELEVANT          AMBIGUOUS
  Use docs           Web search          Use docs + Web search
  ↓                  ↓                   ↓
  LLM → Answer       LLM → Answer        LLM → Answer
```

### Why it's important
Your vector store may be outdated or simply not have the answer.
Without CRAG: system gives a confident but wrong answer using irrelevant docs.
With CRAG: system detects poor docs and searches the web for current information.

### Limitation (what Concept 10 fixes)
The pipeline still decides when to retrieve — the LLM just scores docs, it doesn't control the overall strategy.
→ Concept 10 (Self-RAG) gives the LLM full control over every decision in the pipeline.

In [ ]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries

chunks, vectorstore, embeddings, llm, prompt = setup()

### Step 1 — Doc Relevance Evaluator
Ask the LLM to score each retrieved doc: RELEVANT, IRRELEVANT, or AMBIGUOUS.

In [ ]:
def evaluate_doc(query, doc):
    eval_prompt = (
        f"Is the document below relevant to answer the question?\n\n"
        f"Question: {query}\n\n"
        f"Document: {doc.page_content[:600]}\n\n"
        f"Reply with ONE word only: RELEVANT, IRRELEVANT, or AMBIGUOUS"
    )
    return llm.invoke(eval_prompt).content.strip().upper()

# Test evaluator
query   = TEST_QUERIES[0]
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs    = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(docs):
    score = evaluate_doc(query, doc)
    print(f"Doc {i+1} → {score}: {doc.page_content[:100]}...")

### Step 2 — Web Search Fallback
When retrieved docs are irrelevant, fall back to DuckDuckGo web search (ddgs package).

In [ ]:
def web_search(query, max_results=3):
    try:
        from ddgs import DDGS
        ddgs    = DDGS(timeout=10)
        results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No web results found."
        return "\n\n".join(
            f"[Web] {r.get('title', '')}\n{r.get('body', '')}"
            for r in results
        )
    except Exception as e:
        return f"Web search unavailable: {e}"

### Step 3 — CRAG Pipeline
Retrieve → Evaluate each doc → Decide: use docs / web search / both.

In [ ]:
def crag(query):
    print(f"Query: {query}")

    # Retrieve
    docs = retriever.invoke(query)

    # Evaluate each doc
    relevant   = []
    irrelevant = []
    for doc in docs:
        score = evaluate_doc(query, doc)
        print(f"  Doc evaluation: {score}")
        if score == "RELEVANT":
            relevant.append(doc)
        elif score == "IRRELEVANT":
            irrelevant.append(doc)
        else:  # AMBIGUOUS
            relevant.append(doc)  # treat ambiguous as relevant

    # Decide context source
    if len(relevant) == 0:
        print("  No relevant docs → falling back to web search")
        context = web_search(query)
    elif len(irrelevant) > 0:
        print("  Mixed relevance → using relevant docs + web search")
        context = format_docs(relevant) + "\n\n" + web_search(query)
    else:
        print("  All docs relevant → using knowledge base only")
        context = format_docs(relevant)

    response = llm.invoke(prompt.format_messages(context=context, question=query))
    return response.content

### Test — Standard Queries

In [ ]:
run_queries(crag, label="Concept 9 — CRAG")

### Bonus — Test with a Question NOT in the Knowledge Base
Ask something the LangSmith docs don't have. Watch CRAG fall back to web search.

In [ ]:
out_of_scope = "What is the latest version of Python released in 2025?"
print(crag(out_of_scope))